# Web Scraping dengan Selenium




1. Menyiapkan Selenium dan Chrome.
2. Menentukan URL serta pengaturan scraping.
3. Membuka halaman dan melakukan scroll / klik Load More.
4. Mengambil HTML yang sudah dimuat.
5. Mendeteksi struktur data secara otomatis.
6. Menyimpan hasilnya ke CSV.




In [13]:
# Library yang digunakan
import os
import re
import sys
import time
from datetime import datetime
from urllib.parse import urljoin, urlparse

# Supaya output teks aman di Windows/Jupyter
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Fallback sederhana kalau display() tidak tersedia
try:
    display
except NameError:
    def display(obj):
        print(obj)

# Olah data dan parsing HTML
import pandas as pd
from bs4 import BeautifulSoup

# Selenium untuk membuka dan mengontrol browser
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException
)

print("Python:", sys.version.split()[0])
print("Selenium:", webdriver.__version__)
print("Pandas:", pd.__version__)
print("Library siap digunakan.")


Python: 3.13.2
Selenium: 4.50.0
Pandas: 2.3.3
Library siap digunakan.


In [14]:
# Pengaturan scraping

# URL website yang akan diambil
TARGET_URL = "https://maritime-border-intel.vercel.app/"

# Mode scraping: auto, scroll, atau button
SCRAPE_MODE = "auto"

# Selector elemen data yang ingin diambil
CONTAINER_SELECTOR = "tr.openSectorDetail(1)"

# Selector tombol Load More. Kosongkan jika ingin dideteksi otomatis.
BUTTON_SELECTOR = ""

# Selector tombol Next jika website menggunakan pagination.
PAGINATION_SELECTOR = '.pagination a[rel="next"]'

# Batas proses scraping
MAX_ITEMS =350           # Stop once this many items are collected (Set to None for no limit)
MAX_LOOPS = 25             # Maximum recursive rounds (prevents infinite run)
SCROLL_PAUSE_TIME = 2.0    # Seconds to wait after each scroll/click for JS to render
MAX_UNCHANGED_ROUNDS = 3   # Stop if item count doesn't change after this many rounds

# True = Chrome berjalan di background, False = browser terlihat
HEADLESS = False

# Nama file CSV hasil scraping
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_CSV = f"scraped_data_{TIMESTAMP}.csv"

print("=" * 60)
print(f"Target URL       : {TARGET_URL}")
print(f"Scrape Mode      : {SCRAPE_MODE.upper()}")
print(f"Card Selector    : {CONTAINER_SELECTOR or "[Auto-Detect Active]"}")
print(f"Button Selector  : {BUTTON_SELECTOR or "[Auto-Detect Active]"}")
print(f"Limits           : Max {MAX_ITEMS} items | Max {MAX_LOOPS} loops")
print(f"Headless Mode    : {HEADLESS}")
print(f"Output File      : {OUTPUT_CSV}")
print("=" * 60)


Target URL       : https://maritime-border-intel.vercel.app/
Scrape Mode      : AUTO
Card Selector    : tr.openSectorDetail(1)
Button Selector  : [Auto-Detect Active]
Limits           : Max 350 items | Max 25 loops
Headless Mode    : False
Output File      : scraped_data_20261008_093701.csv


In [15]:
# Membuat Chrome driver

def create_stealth_driver(headless: bool = True) -> webdriver.Chrome:
    """
    Membuat Chrome WebDriver untuk proses scraping.
    """
    options = Options()

    if headless:
        # Mode headless Chrome
        options.add_argument("--headless=new")

    # Beberapa pengaturan browser
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    # Pengaturan supaya Chrome lebih stabil
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--start-maximized")

    # User-Agent browser
    user_agent = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    )
    options.add_argument(f"user-agent={user_agent}")

    # Jalankan Chrome
    driver = webdriver.Chrome(options=options)

    # Pengaturan tambahan pada browser
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {
            "source": """
                Object.defineProperty(navigator, 'webdriver', {
                    get: () => undefined
                });
                window.chrome = {
                    runtime: {}
                };
                Object.defineProperty(navigator, 'plugins', {
                    get: () => [1, 2, 3, 4, 5]
                });
                Object.defineProperty(navigator, 'languages', {
                    get: () => ['en-US', 'en', 'id']
                });
            """
        }
    )

    return driver

print("Fungsi Chrome driver siap.")


Fungsi Chrome driver siap.


In [16]:
# Fungsi untuk scroll dan Load More

def smooth_scroll_down(driver, step_px: int = 450, pause_sec: float = 0.12):
    """
    Scroll sedikit demi sedikit supaya data lazy loading ikut muncul.
    """
    current_pos = driver.execute_script("return window.pageYOffset;")
    total_height = driver.execute_script(
        "return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight);"
    )

    while current_pos < total_height:
        current_pos += step_px
        driver.execute_script(f"window.scrollTo(0, {current_pos});")
        time.sleep(pause_sec)
        total_height = driver.execute_script(
            "return Math.max(document.body.scrollHeight, document.documentElement.scrollHeight);"
        )

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")


def get_current_item_count(driver, container_selector: str = "") -> int:
    """Menghitung jumlah data yang sedang tampil."""
    try:
        if container_selector:
            return len(driver.find_elements(By.CSS_SELECTOR, container_selector))

        return len(driver.find_elements(
            By.XPATH,
            "//article | //div[contains(@class, 'card') or "
            "contains(@class, 'item') or contains(@class, 'quote') or "
            "contains(@class, 'post')]"
        ))
    except Exception:
        return 0


def find_and_click_load_more(driver, custom_selector: str = "") -> bool:
    """Mencari dan menekan tombol Load More jika tersedia."""
    if custom_selector:
        try:
            buttons = driver.find_elements(By.CSS_SELECTOR, custom_selector)
            for btn in buttons:
                try:
                    if btn.is_displayed() and btn.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView({behavior:'smooth',block:'center'});", btn)
                        time.sleep(0.3)
                        driver.execute_script("arguments[0].click();", btn)
                        return True
                except Exception:
                    continue
        except Exception:
            pass

    try:
        candidates = driver.find_elements(By.XPATH, "//button | //a | //*[@role='button']")
        prefixes = (
            "load more", "show more", "view more", "see more",
            "muat lebih", "lihat lebih", "tampilkan lebih",
            "baca lebih banyak", "selanjutnya"
        )
        for el in candidates:
            try:
                if not el.is_displayed() or not el.is_enabled():
                    continue
                text_value = re.sub(r"\s+", " ", (el.text or "").strip().lower())
                aria = re.sub(r"\s+", " ", (el.get_attribute("aria-label") or "").strip().lower())
                title = re.sub(r"\s+", " ", (el.get_attribute("title") or "").strip().lower())
                values = (text_value, aria, title)
                matched = any(any(v == p or v.startswith(p + " " ) or v.startswith(p + "(") for p in prefixes) for v in values if v)
                if not matched:
                    continue
                driver.execute_script("arguments[0].scrollIntoView({behavior:'smooth',block:'center'});", el)
                time.sleep(0.4)
                try:
                    el.click()
                except Exception:
                    driver.execute_script("arguments[0].click();", el)
                return True
            except (StaleElementReferenceException, ElementClickInterceptedException):
                continue
            except Exception:
                continue
    except Exception:
        pass
    return False

def wait_for_item_count_change(
    driver,
    container_selector: str,
    previous_count: int,
    timeout: float = 12.0,
    poll: float = 0.25
) -> int:
    """
    Menunggu sampai jumlah data bertambah.
    """
    end_time = time.time() + timeout
    latest = previous_count

    while time.time() < end_time:
        latest = get_current_item_count(driver, container_selector)

        if latest > previous_count:
            return latest

        time.sleep(poll)

    return latest


def recursive_lazy_scrape(
    driver, mode: str = "auto", container_selector: str = "",
    button_selector: str = "", max_loops: int = 25, max_items: int = 180,
    pause_time: float = 2.0, max_unchanged: int = 3, **_kwargs
) -> str:
    """Mengambil data sampai batas item, loop, atau data sudah habis.ot counted
    as an unchanged round.
    """
    unchanged_rounds = 0
    current_count = get_current_item_count(driver, container_selector)

    for loop in range(1, max_loops + 1):
        current_count = get_current_item_count(driver, container_selector)
        print(f"🔄 [Loop {loop:02d}/{max_loops:02d}] Target items in DOM: {current_count:4d}")

        if max_items and current_count >= max_items:
            print(f"🎯 [Target Achieved] {current_count} items >= target {max_items}.")
            return driver.page_source

        if mode in ["auto", "scroll"]:
            smooth_scroll_down(driver)
            time.sleep(pause_time)
            current_count = get_current_item_count(driver, container_selector)
            if max_items and current_count >= max_items:
                print(f"🎯 [Target Achieved] {current_count} items >= target {max_items}.")
                return driver.page_source

        before_click = get_current_item_count(driver, container_selector)
        clicked = False
        if mode in ["auto", "button"]:
            clicked = find_and_click_load_more(driver, button_selector)

        if clicked:
            print("   🔘 Load More clicked. Waiting for new target items...")
            after_click = wait_for_item_count_change(
                driver, container_selector, before_click,
                timeout=max(10.0, pause_time + 8.0)
            )
            if after_click > before_click:
                print(f"   ✅ Target items increased: {before_click} → {after_click}")
                unchanged_rounds = 0
            else:
                unchanged_rounds += 1
                print(f"   ⚠️ Load More did not increase target count: {before_click} → {after_click}")
        else:
            unchanged_rounds += 1
            print("   ⚠️ Load More control not found/clickable.")

        if max_items and get_current_item_count(driver, container_selector) >= max_items:
            print(f"🎯 [Target Achieved] Reached {max_items} items.")
            return driver.page_source

        if unchanged_rounds >= max_unchanged:
            print(f"🏁 [Completed] No target growth after {max_unchanged} unsuccessful attempts.")
            return driver.page_source

        time.sleep(pause_time)

    print(f"🛑 [Reached Limit] Maximum loops ({max_loops}) reached.")
    return driver.page_source

print("✅ Recursive scraping engine compiled!")


✅ Recursive scraping engine compiled!


In [17]:
# Fungsi untuk membaca dan menyusun data HTML

def clean_text(text: str) -> str:
    """Membersihkan spasi dan karakter yang tidak perlu."""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_column_name(name: str, fallback: str = "column") -> str:
    """Mengubah nama field menjadi nama kolom yang lebih rapi."""
    name = clean_text(name)
    name = re.sub(r"[^\w\s-]", "", name, flags=re.UNICODE)
    name = re.sub(r"[-\s]+", "_", name).strip("_")
    return name or fallback


def make_unique_columns(columns):
    """Membuat nama kolom yang sama menjadi unik."""
    used = {}
    result = []
    for col in columns:
        base = normalize_column_name(col)
        used[base] = used.get(base, 0) + 1
        result.append(base if used[base] == 1 else f"{base}_{used[base]}")
    return result


def auto_detect_card_selector(soup: BeautifulSoup):
    """Mencari pola card secara otomatis jika selector kosong."""
    candidates = [
        "a[class*='report-card']",
        "div[class*='report-card']",
        "div[class*='quote']",
        "div[class*='card']",
        "div[class*='product']",
        "div[class*='item']",
        "div[class*='post']",
        "div[class*='article']",
        "article",
        "li[class*='item']",
        "li[class*='product']",
        "li[class*='card']",
    ]

    for sel in candidates:
        try:
            elements = soup.select(sel)
            valid = [
                e for e in elements
                if len(clean_text(e.get_text())) > 10 or e.find("a")
            ]
            if len(valid) >= 3:
                return sel, valid
        except Exception:
            continue

    class_counter = {}
    for el in soup.find_all(["div", "article", "li", "a"]):
        classes = el.get("class", [])
        if classes:
            key = "." + ".".join(classes)
            class_counter[key] = class_counter.get(key, 0) + 1

    sorted_classes = sorted(
        [k for k, v in class_counter.items() if v >= 3],
        key=lambda k: class_counter[k],
        reverse=True
    )

    if sorted_classes:
        top_sel = sorted_classes[0]
        return top_sel, soup.select(top_sel)

    return "body", [soup.body] if soup.body else []


def extract_html_table_records(soup: BeautifulSoup):
    """Mengambil data dari tabel HTML jika ditemukan."""
    tables = soup.find_all("table")
    best_records = []
    best_headers = []

    for table in tables:
        rows = table.find_all("tr")
        if len(rows) < 2:
            continue

        header_cells = rows[0].find_all(["th", "td"])
        headers = [
            clean_text(c.get_text(" ", strip=True))
            for c in header_cells
        ]
        headers = [h for h in headers if h]

        if len(headers) < 2:
            continue

        records = []
        for row in rows[1:]:
            cells = row.find_all(["td", "th"])
            values = [
                clean_text(c.get_text(" ", strip=True))
                for c in cells
            ]

            if not any(values):
                continue

            if len(values) < len(headers):
                values += [""] * (len(headers) - len(values))

            record = dict(
                zip(make_unique_columns(headers), values[:len(headers)])
            )
            records.append(record)

        if len(records) > len(best_records):
            best_records = records
            best_headers = headers

    if best_records:
        print(
            f"📋 HTML table detected -> {len(best_records)} rows | "
            f"Columns: {best_headers}"
        )

    return best_records


def extract_json_records(soup: BeautifulSoup):
    """
    Mencari data JSON yang ada di dalam halaman.
    Data navigasi tidak dipakai sebagai dataset.
    """
    import json

    candidates = []

    for script in soup.find_all("script"):
        script_type = (script.get("type") or "").lower()
        raw = script.string or script.get_text()

        if not raw or len(raw.strip()) < 10:
            continue

        if "json" not in script_type and not raw.lstrip().startswith(("[", "{")):
            continue

        try:
            obj = json.loads(raw.strip())
        except Exception:
            continue

        candidates.append(obj)

    def collect_list_of_dicts(obj):
        found = []

        if isinstance(obj, list) and obj and all(
            isinstance(x, dict) for x in obj
        ):
            found.append(obj)

        elif isinstance(obj, dict):
            for key, value in obj.items():
                # Ignore obvious schema/navigation metadata branches.
                if key in {
                    "@context", "@graph", "breadcrumb", "potentialAction"
                }:
                    continue
                found.extend(collect_list_of_dicts(value))

        return found

    ignored_types = {
        "ListItem",
        "BreadcrumbList",
        "WebSite",
        "WebPage",
        "Organization",
        "ImageObject",
        "SearchAction",
        "SiteNavigationElement",
    }

    possible = []

    for obj in candidates:
        for records in collect_list_of_dicts(obj):
            if len(records) < 2:
                continue

            cleaned = []

            for item in records:
                item_type = item.get("@type") or item.get("type")

                if isinstance(item_type, list):
                    item_type = item_type[0] if item_type else ""

                if str(item_type) in ignored_types:
                    continue

                record = {}
                for key, value in item.items():
                    if key.startswith("@"):
                        continue

                    if isinstance(value, (dict, list)):
                        value = str(value)

                    value = clean_text(value)

                    if value != "":
                        record[normalize_column_name(key)] = value

                if record:
                    cleaned.append(record)

            if len(cleaned) >= 2:
                # Reject the classic breadcrumb shape even if @type was absent.
                all_keys = set().union(*(r.keys() for r in cleaned))
                breadcrumb_keys = {"type", "position", "name", "item"}

                if all_keys.issubset(breadcrumb_keys):
                    continue

                possible.append(cleaned)

    if possible:
        # Prefer the candidate with the largest number of records.
        best = max(possible, key=len)
        print(
            f"🧩 Structured JSON detected -> {len(best)} records | "
            f"Columns: {list(best[0].keys())}"
        )
        return best

    return []


def extract_key_value_fields(card):
    """Extract explicit label/value pairs from a card."""
    record = {}

    for dt in card.find_all("dt"):
        dd = dt.find_next_sibling("dd")
        if dd:
            key = normalize_column_name(dt.get_text(" ", strip=True))
            value = clean_text(dd.get_text(" ", strip=True))
            if key and value:
                record[key] = value

    for label in card.find_all("label"):
        value_node = label.find_next_sibling()
        if value_node:
            key = normalize_column_name(label.get_text(" ", strip=True))
            value = clean_text(value_node.get_text(" ", strip=True))
            if key and value:
                record[key] = value

    label_nodes = card.select(
        ".field-label, .label, .key, [data-label]"
    )

    for label in label_nodes:
        key = normalize_column_name(
            label.get("data-label") or label.get_text(" ", strip=True)
        )
        sibling = label.find_next_sibling()

        if sibling:
            value = clean_text(sibling.get_text(" ", strip=True))
            if key and value and key not in record:
                record[key] = value

    return record


def extract_generic_card_record(
    card,
    base_url: str,
    idx: int
) -> dict:
    """
    Generic fallback for card pages.

    The output is based on information actually present in the card.
    It does not impose the old 9-column schema.
    """
    record = extract_key_value_fields(card)

    title = ""

    for heading in card.find_all(
        ["h1", "h2", "h3", "h4", "h5", "strong", "b"]
    ):
        value = clean_text(heading.get_text())
        if len(value) > 3:
            title = value
            break

    if not title:
        first_a = card.find("a", href=True)
        if first_a:
            title = clean_text(first_a.get_text())

    link = ""
    a_tag = card.find("a", href=True)

    if a_tag:
        link = urljoin(base_url, a_tag["href"])

    image_url = ""
    img = card.find("img")

    if img:
        for attr in [
            "data-src",
            "data-lazy-src",
            "data-original",
            "srcset",
            "src",
        ]:
            raw = img.get(attr)

            if raw:
                if attr == "srcset":
                    raw = raw.split(",")[0].split()[0]

                image_url = urljoin(base_url, raw)
                break

    if title and "title" not in record:
        record["title"] = title

    if link and "link" not in record:
        record["link"] = link

    if image_url and "image_url" not in record:
        record["image_url"] = image_url

    text = clean_text(card.get_text(" ", strip=True))

    if text and "text" not in record:
        record["text"] = text

    return record


def extract_data_from_cards(
    html_content: str,
    base_url: str,
    container_selector: str = ""
) -> list:
    """
    Extract data using the user's selector FIRST.

    This is important:
    If the user explicitly supplies a selector such as
    `a.report-card`, that selector defines the dataset.

    We do NOT allow unrelated JSON-LD breadcrumbs/navigation to override it.

    Priority:
      1. Explicit user selector.
      2. HTML table.
      3. Useful structured JSON.
      4. Automatic card detection.
    """
    soup = BeautifulSoup(html_content, "html.parser")

    # ------------------------------------------------------------------
    # 1. USER SELECTOR HAS PRIORITY
    # ------------------------------------------------------------------
    if container_selector:
        cards = soup.select(container_selector)

        print(
            f"🎯 User selector: '{container_selector}' -> "
            f"Found {len(cards)} elements"
        )

        if cards:
            records = []

            for idx, card in enumerate(cards, start=1):
                record = extract_generic_card_record(
                    card,
                    base_url,
                    idx
                )

                if record:
                    records.append(record)

            print(
                f"📦 Extracted from user selector -> "
                f"{len(records)} records"
            )

            if records:
                return records

        print(
            "⚠️ User selector returned no usable records. "
            "Falling back to automatic schema detection..."
        )

    # ------------------------------------------------------------------
    # 2. HTML TABLE
    # ------------------------------------------------------------------
    table_records = extract_html_table_records(soup)

    if table_records:
        return table_records

    # ------------------------------------------------------------------
    # 3. STRUCTURED JSON
    # ------------------------------------------------------------------
    json_records = extract_json_records(soup)

    if json_records:
        return json_records

    # ------------------------------------------------------------------
    # 4. AUTO CARD DETECTION
    # ------------------------------------------------------------------
    used_selector, cards = auto_detect_card_selector(soup)

    print(
        f"📦 Auto card extraction using: '{used_selector}' -> "
        f"Found {len(cards)} cards"
    )

    records = []

    for idx, card in enumerate(cards, start=1):
        record = extract_generic_card_record(
            card,
            base_url,
            idx
        )

        if record:
            records.append(record)

    return records


def align_dynamic_schema(records: list) -> pd.DataFrame:
    """
    Create a DataFrame using the union of fields found across records.

    No fixed schema is imposed.
    """
    if not records:
        return pd.DataFrame()

    columns = []

    for record in records:
        for key in record.keys():
            key = normalize_column_name(key)

            if key not in columns:
                columns.append(key)

    normalized_records = []

    for record in records:
        normalized_records.append({
            normalize_column_name(k): v
            for k, v in record.items()
        })

    return pd.DataFrame(
        normalized_records,
        columns=columns
    )


print("✅ Dynamic schema parser compiled!")


✅ Dynamic schema parser compiled!


In [18]:
# Jalankan scraping dan simpan hasil ke CSV

driver = None
try:
    print("Membuka Chrome...")
    driver = create_stealth_driver(headless=HEADLESS)

    print(f"Membuka URL: {TARGET_URL}")
    driver.get(TARGET_URL)
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.TAG_NAME, "body"))
    )
    time.sleep(SCROLL_PAUSE_TIME)

    print("\n" + "=" * 60)
    print("Mulai proses scraping")
    print("=" * 60)
    final_html = recursive_lazy_scrape(
        driver=driver,
        mode=SCRAPE_MODE,
        container_selector=CONTAINER_SELECTOR,
        button_selector=BUTTON_SELECTOR,
        max_loops=MAX_LOOPS,
        max_items=MAX_ITEMS,
        pause_time=SCROLL_PAUSE_TIME,
        max_unchanged=MAX_UNCHANGED_ROUNDS
    )

    print("=" * 60)
    print("Proses scraping selesai.\n")

    print("Membaca struktur data dan mengambil record...")
    data_records = extract_data_from_cards(
        html_content=final_html,
        base_url=TARGET_URL,
        container_selector=CONTAINER_SELECTOR
    )

    if not data_records:
        print("Tidak ada data yang berhasil diambil. Cek URL atau selector.")
    else:
        # Kolom mengikuti data yang ditemukan.
        df = align_dynamic_schema(data_records)

        # Hapus duplikat berdasarkan kolom identitas yang tersedia.
        dedup_candidates = [
            "ID_Log", "id", "ID", "link", "url", "title", "name", "text"
        ]
        dedup_col = next((c for c in dedup_candidates if c in df.columns), None)
        initial_len = len(df)
        if dedup_col:
            df = df.drop_duplicates(subset=[dedup_col]).copy()
            print(f"Data duplikat berdasarkan '{dedup_col}': {initial_len - len(df)} baris dihapus.")

        if MAX_ITEMS and len(df) > MAX_ITEMS:
            df = df.iloc[:MAX_ITEMS].copy()

        # Simpan CSV tanpa index tambahan.
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

        abs_path = os.path.abspath(OUTPUT_CSV)
        print("\n" + "=" * 60)
        print("DATASET BERHASIL DIBUAT")
        print(f"File Name      : {OUTPUT_CSV}")
        print(f"Full Path      : {abs_path}")
        print(f"Total Rows     : {len(df)}")
        print(f"Total Columns  : {len(df.columns)}")
        print(f"Detected Schema: {list(df.columns)}")
        print("=" * 60 + "\n")

        display(df.head(10))

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

finally:
    if driver:
        print("\nMenutup Chrome...")
        driver.quit()
        print("Chrome sudah ditutup.")


Membuka Chrome...
Membuka URL: https://maritime-border-intel.vercel.app/

Mulai proses scraping
🔄 [Loop 01/25] Target items in DOM:    0
   ⚠️ Load More control not found/clickable.
🔄 [Loop 02/25] Target items in DOM:    0
   ⚠️ Load More control not found/clickable.
🔄 [Loop 03/25] Target items in DOM:    0
   ⚠️ Load More control not found/clickable.
🏁 [Completed] No target growth after 3 unsuccessful attempts.
Proses scraping selesai.

Membaca struktur data dan mengambil record...
Error: Invalid character '(' position 19
  line 1:
tr.openSectorDetail(1)
                   ^

Menutup Chrome...


Traceback (most recent call last):
  File "C:\Users\ossas\AppData\Local\Temp\ipykernel_18332\2520750334.py", line 33, in <module>
    data_records = extract_data_from_cards(
        html_content=final_html,
        base_url=TARGET_URL,
        container_selector=CONTAINER_SELECTOR
    )
  File "C:\Users\ossas\AppData\Local\Temp\ipykernel_18332\3183308461.py", line 378, in extract_data_from_cards
    cards = soup.select(container_selector)
  File "c:\Users\ossas\AppData\Local\Programs\Python\Python313\Lib\site-packages\bs4\element.py", line 2116, in select
    return self.css.select(selector, namespaces, limit, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ossas\AppData\Local\Programs\Python\Python313\Lib\site-packages\bs4\css.py", line 162, in select
    self.api.select(
    ~~~~~~~~~~~~~~~^
        select, self.tag, self._ns(namespaces, select), limit, flags,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        **

Chrome sudah ditutup.


## Catatan tentang struktur data




Urutan sederhananya:
- cari card atau elemen data
- ambil teks/field yang tersedia
- rapikan nama kolom
- gabungkan record
- hapus duplikat jika ada
- simpan sebagai CSV

